In [4]:
! pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 18.2 MB/s eta 0:00:000:00:010:00:0101


In [5]:
import cv2
import numpy as np
from PIL import Image
from scipy.linalg import sqrtm
import torch
from torchvision.models import inception_v3
from torchvision.transforms import transforms

# Load the pretrained Inception model
model = inception_v3(pretrained=True, transform_input=False).eval()

# Function to extract frames from a video
def extract_frames(video_path, max_frames=100):
    frames = []
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    success, frame = cap.read()
    while success and frame_count < max_frames:
        # Convert BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(frame_rgb))
        success, frame = cap.read()
        frame_count += 1
    cap.release()
    return frames

# Function to extract features
def extract_features(images, model):
    preprocess = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    images = torch.stack([preprocess(img) for img in images])
    with torch.no_grad():
        features = model(images)
    return features.numpy()

# Calculate FID
def calculate_fid(real_features, generated_features):
    mu_real, sigma_real = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
    mu_gen, sigma_gen = np.mean(generated_features, axis=0), np.cov(generated_features, rowvar=False)

    # Regularize covariance matrices
    epsilon = 1e-6
    sigma_real += epsilon * np.eye(sigma_real.shape[0])
    sigma_gen += epsilon * np.eye(sigma_gen.shape[0])

    # Compute Fréchet distance
    diff = mu_real - mu_gen
    covmean = sqrtm(sigma_real @ sigma_gen)

    # Handle numerical instability
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma_real + sigma_gen - 2 * covmean)
    return fid



def compute_fid(video_a_path, video_b_path):
    # Extract frames from both videos
    frames_a = extract_frames(video_a_path, max_frames=100)
    frames_b = extract_frames(video_b_path, max_frames=100)
    
    # Extract features using the Inception model
    features_a = extract_features(frames_a, model)
    features_b = extract_features(frames_b, model)
    
    # Calculate FID
    fid_score = calculate_fid(features_a, features_b)
    print(f"FID Score: {fid_score}")


/Users/hims/anaconda3/envs/demo1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/hims/anaconda3/envs/demo1/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/hims/anaconda3/envs/demo1/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
# Paths to the video files
compute_fid(video_a_path = "/Users/hims/anjali_git_repos/data-298/data-298B/demo1/web-app/assets/sample/video3.mp4",
            video_b_path = "/Users/hims/anjali_git_repos/data-298/data-298B/demo1/web-app/assets/sample/video3.mp4")



FID Score: 2.1938250313602303e-07


In [7]:
# Paths to the video files
compute_fid(video_a_path = "/Users/hims/anjali_git_repos/data-298/data-298B/demo1/web-app/assets/sample/nike.mp4",
            video_b_path = "/Users/hims/anjali_git_repos/data-298/data-298B/demo1/web-app/assets/sample/nike.mp4")



FID Score: 2.851179801768855e-07


In [8]:
# Paths to the video files
compute_fid(video_a_path = "/Users/hims/anjali_git_repos/data-298/data-298B/demo1/web-app/assets/sample/nike.mp4",
            video_b_path = "/Users/hims/anjali_git_repos/data-298/data-298B/demo1/web-app/assets/sample/smart_watch.mp4")



FID Score: 1884.42306243451


In [12]:
import torch
import clip
from PIL import Image
import cv2

# Load CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/32", device=device)

def calculate_clip_similarity_from_video(text, video_path, max_frames=100):
    """
    Calculate the CLIP similarity between a text prompt and a video.

    Args:
        text (str): The text prompt to compare against.
        video_path (str): Path to the video file.
        max_frames (int): Maximum number of frames to process from the video.

    Returns:
        float: The CLIP similarity score between the text and the video frames.
    """
    # Extract frames from the video
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = 0

    while frame_count < max_frames:
        success, frame = cap.read()
        if not success:
            break
        # Convert BGR to RGB for PIL
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(frame_rgb))
        frame_count += 1

    cap.release()

    # If no frames were extracted, return similarity as 0
    if not frames:
        print("No frames extracted from video.")
        return 0.0

    # Preprocess images and text
    image_tensors = torch.stack([preprocess(img).to(device) for img in frames])
    text_tokens = clip.tokenize([text]).to(device)

    # Calculate CLIP features
    with torch.no_grad():
        image_features = clip_model.encode_image(image_tensors)
        text_features = clip_model.encode_text(text_tokens)

    # Normalize features and compute cosine similarity
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    similarity = torch.matmul(image_features, text_features.T).mean().item()

    return similarity


# Example usage
text_prompt = "A cat playing in the garden"
video_path = "a.mp4"  # Path to your video file
clip_similarity = calculate_clip_similarity_from_video("A man running in a snowy day wearing track pants and nike hoodie. He running on the road, then suddenly a billboard comes up where it shows the nike log in black and white color.",
                                                       "/Users/hims/anjali_git_repos/data-298/data-298B/demo1/web-app/assets/sample/nike.mp4" )
print(f"CLIP Similarity: {clip_similarity}")


AttributeError: module 'clip' has no attribute 'load'

In [13]:
! pip uninstall clip

Found existing installation: clip 0.2.0
Uninstalling clip-0.2.0:
  Would remove:
    /Users/hims/anaconda3/envs/demo1/bin/clip
    /Users/hims/anaconda3/envs/demo1/lib/python3.10/site-packages/clip-0.2.0.dist-info/*
    /Users/hims/anaconda3/envs/demo1/lib/python3.10/site-packages/clip/*
Proceed (Y/n)? ^C
ERROR: Operation cancelled by user


In [14]:
! pip install moviepy


  Preparing metadata (setup.py) ... done
  Using cached proglog-0.1.10-py3-none-any.whl.metadata (639 bytes)
  Created wheel for imageio_ffmpeg: filename=imageio_ffmpeg-0.5.1-py3-none-any.whl size=16724 sha256=3c3857be34dd68c869f0bb950ee3a390e35fb00a593ed48b1f08093443d53d0e
  Stored in directory: /Users/hims/Library/Caches/pip/wheels/b1/02/a8/11be66aeef7c8f041b20a8110050ade155202ce9fa4a2809b2
Successfully built imageio_ffmpeg


In [15]:
from moviepy.editor import VideoFileClip, concatenate_videoclips

def merge_videos(video_paths, output_path):
    """
    Merges multiple MP4 videos into one.

    Args:
        video_paths (list): List of paths to the MP4 videos to be merged.
        output_path (str): Path for the output merged video.

    Returns:
        None
    """
    try:
        # Load video clips
        clips = [VideoFileClip(video) for video in video_paths]

        # Concatenate video clips
        merged_clip = concatenate_videoclips(clips, method="compose")

        # Write the final output
        merged_clip.write_videofile(output_path, codec="libx264", audio_codec="aac")

        print(f"Merged video saved to {output_path}")
    except Exception as e:
        print(f"Error occurred: {e}")

# Example usage
video_list = ["/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_in_h.mp4", 
              "/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_in_a.mp4", 
              "/Users/hims/Downloads/CogVideo/scene_1___a_black_merscdese_be.mp4",
              "/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_glid.mp4"]  # Replace with your video file paths
output_file = "/Users/hims/Downloads/CogVideo/merc_merged_video.mp4"
merge_videos(video_list, output_file)


ModuleNotFoundError: No module named 'moviepy.editor'

In [29]:
# import subprocess
# import os

# def merge_videos_ffmpeg(video_files, output_file):
#     """
#     Merge multiple MP4 videos into one using ffmpeg.

#     Args:
#         video_files (list of str): List of video file paths to merge.
#         output_file (str): Path to save the merged video file.
#     """
#     with open('filelist.txt', 'w') as file:
#         for video_file in video_files:
#             file.write(f"file '{video_file}'\n")

#     if os.path.exists(output_file):
#         os.remove(output_file)
#     command = [
#         'ffmpeg',
#         '-f', 'concat',
#         '-safe', '0',
#         '-i', 'filelist.txt',
#         '-c', 'copy',
#         output_file
#     ]

#     # Execute the ffmpeg command
#     subprocess.run(command, check=True)

#     # Clean up the temporary file list
#     subprocess.run(['rm', 'filelist.txt'])

# # # Example usage
# # video_files = ['video1.mp4', 'video2.mp4', 'video3.mp4']
# # output_file = 'merged_output.mp4'
# # merge_videos_ffmpeg(video_files, output_file)

# video_list = ["/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_in_h.mp4", 
#               "/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_in_a.mp4", 
#               "/Users/hims/Downloads/CogVideo/black_mercesdese_benz_suv_and_.mp4",
#               "/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_glid.mp4"]  # Replace with your video file paths
# output_file = "/Users/hims/Downloads/CogVideo/merc_merged_video.mp4"
# merge_videos_ffmpeg(video_list, output_file)


ffmpeg version 7.0.2 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 15.0.0 (clang-1500.3.9.4)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.0.2_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex 

In [31]:
# def merge_gifs_to_mp4(gif_files, output_file):
#     """
#     Merge multiple GIF files into one MP4 video using ffmpeg.

#     Args:
#         gif_files (list of str): List of GIF file paths to merge.
#         output_file (str): Path to save the merged MP4 video file.
#     """
#     with open('filelist.txt', 'w') as file:
#         for gif_file in gif_files:
#             file.write(f"file '{gif_file}'\n")

#     if os.path.exists(output_file):
#         os.remove(output_file)

#     # Concatenate GIFs and convert to MP4 with re-encoding for compatibility and smooth playback
#     command = [
#         'ffmpeg',
#         '-f', 'concat',
#         '-safe', '0',
#         '-i', 'filelist.txt',
#         '-vsync', 'vfr',  # Use variable frame rate to handle different frame rates in GIFs
#         '-pix_fmt', 'yuv420p',  # Set pixel format to yuv420p for broad compatibility
#         '-c:v', 'libx264',  # Encode video stream with x264 (H.264)
#         '-preset', 'fast',  # Adjust encoding speed (trade-off between speed and compression)
#         '-crf', '23',  # Constant Rate Factor (trade-off between quality and file size)
#         output_file
#     ]

#     # Execute the ffmpeg command
#     subprocess.run(command, check=True)

#     # Clean up the temporary file list
#     subprocess.run(['rm', 'filelist.txt'])


# video_list = ["/Users/hims/Downloads/Tune-A-Video/a_cat_is_playing_in_grass.gif", 
#               "/Users/hims/Downloads/Tune-A-Video/scene_1___cat_is_playing_in_gr.gif"]  # Replace with your video file paths

# output_file = "/Users/hims/Downloads/Tune-A-Video/merged.mp4"
# merge_gifs_to_mp4(video_list, output_file)

ffmpeg version 7.0.2 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 15.0.0 (clang-1500.3.9.4)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.0.2_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex 

In [56]:
import subprocess
import os

def merge_and_convert_files(media_files, output_full_path):
    """
    Merge media files (GIFs or MP4s) into a single file and convert to both MP4 and GIF.
    Automatically extracts the base path from the provided full output path to generate outputs.

    Args:
        media_files (list of str): List of media file paths to merge.
        output_full_path (str): Full path to save the merged video file with an extension.
    """
    # Extract base path and extension for output files
    output_base, output_ext = os.path.splitext(output_full_path)

    # Write the file list to a text file for ffmpeg processing
    with open('filelist.txt', 'w') as file:
        for media_file in media_files:
            file.write(f"file '{media_file}'\n")

    # Determine the file type from the first file's extension
    file_extension = os.path.splitext(media_files[0])[1].lower()

    # Output files
    output_mp4 = f"{output_base}.mp4"
    output_gif = f"{output_base}.gif"

    if os.path.exists(output_mp4): os.remove(output_mp4)
    if os.path.exists(output_gif): os.remove(output_gif)

    if file_extension == '.gif':
        # Process GIF files to MP4
        convert_command = [
            'ffmpeg',
            '-f', 'concat',
            '-safe', '0',
            '-i', 'filelist.txt',
            '-vsync', 'vfr',  # Handle different frame rates in GIFs
            '-pix_fmt', 'yuv420p',  # Ensure broad compatibility
            '-c:v', 'libx264',
            '-preset', 'fast',
            '-crf', '23',
            output_mp4
        ]
    else:  # Assume MP4 or other video formats
        # Simply concatenate videos for MP4 output
        convert_command = [
            'ffmpeg',
            '-f', 'concat',
            '-safe', '0',
            '-i', 'filelist.txt',
            '-c', 'copy',
            output_mp4
        ]

    # Execute the ffmpeg command to create MP4
    subprocess.run(convert_command, check=True)

    # Convert the output MP4 to GIF for a unified output
    gif_command = [
        'ffmpeg',
        '-i', output_mp4,
        '-vf', "fps=10,scale=320:-1:flags=lanczos",  # Adjust fps and scale according to needs
        '-loop', '0',
        output_gif
    ]

    # Execute the ffmpeg command to create GIF
    subprocess.run(gif_command, check=True)

    # Clean up the temporary file list
    # subprocess.run(['rm', 'filelist.txt'])




# video_list = ["/Users/hims/Downloads/Tune-A-Video/a_cat_is_playing_in_grass.gif", 
#               "/Users/hims/Downloads/Tune-A-Video/scene_1___cat_is_playing_in_gr.gif",
#               "/tmp/output_video.gif"]  # Replace with your video file paths

# output_file = "/Users/hims/Downloads/Tune-A-Video/merged.mp4"
# merge_and_convert_files(video_list, output_file)


In [57]:

video_list = ["/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_in_h.mp4", 
              "/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_in_a.mp4", 
              "/Users/hims/Downloads/CogVideo/black_mercesdese_benz_suv_and_.mp4",
              "/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_glid.mp4",
              "/tmp/output_video.mp4"]  # Replace with your video file paths

output_file = "/Users/hims/Downloads/CogVideo/merc_merged_video.mp4"
merge_and_convert_files(video_list, output_file)


ffmpeg version 7.0.2 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 15.0.0 (clang-1500.3.9.4)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.0.2_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex 

In [61]:
import subprocess
import os
import json

def get_media_dimensions(media_path):
    try:
        # Prepare the ffprobe command to get width and height of the video
        command = [
            'ffprobe',
            '-v', 'error',  # Hide all error messages except critical ones
            '-select_streams', 'v:0',  # Select the first video stream
            '-show_entries', 'stream=width,height',  # Show width and height entries
            '-of', 'json',  # Output in JSON format
            media_path
        ]

        # Execute the ffprobe command and parse the output as JSON
        result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        output = json.loads(result.stdout)
        # Extract width and height from the ffprobe output
        width = output['streams'][0]['width']
        height = output['streams'][0]['height']

        return width, height
    except Exception as e:
        print(f"An error occurred: {e}")
        return 680, 480  # Default dimensions in case of an error

def create_text_video(text, output_full_path, duration=1, media_path=None):
    output_base, output_ext = os.path.splitext(output_full_path)
    output_ext = output_ext.lower()

    # Determine the codec and output file path based on the file extension
    if output_ext == '.gif':
        codec = ["-c:v", "gif", "-f", "gif"]
        output_path = f"{output_base}.gif"
    else:
        codec = ["-c:v", "libx264", "-pix_fmt", "yuv420p", "-tune", "stillimage", "-shortest"]
        output_path = f"{output_base}.mp4"

    # Check if the output file already exists and delete it if it does
    if os.path.exists(output_path):
        os.remove(output_path)

    width, height = get_media_dimensions(media_path)
    font_size = max(height // 20, 20)  # Calculate font size as 1/20 of the height, with a minimum size of 20

    # Set the background filter
    input_filter = f'color=c=black:s={width}x{height}'

    # Set the filter complex for drawing text
    filter_complex = f"drawtext=text='{text}':fontcolor=white:fontsize={font_size}:x=(w-text_w)/2:y=(h-text_h)/2"

    command = [
        'ffmpeg',
        '-f', 'lavfi',
        '-i', input_filter,
        '-vf', filter_complex,
        '-t', str(duration)
    ] + codec + [output_path]

    # Execute the command and capture output
    try:
        process = subprocess.run(command, check=True, text=True, capture_output=True)
        print("STDOUT:")
        print(process.stdout)
        print("STDERR:")
        print(process.stderr)
    except subprocess.CalledProcessError as e:
        print("Error:")
        print(e.stderr)

# Usage example
text_to_display = "Mercedes Benz\nGreat Driving Pleasure."  # The text you want to display
output_file_path = "/tmp/output_video.gif"  # Change the extension to .gif for GIF output
sample_media_path = "/Users/hims/Downloads/Tune-A-Video/merged.gif"
# sample_media_path = "/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_glid.mp4"
create_text_video(text_to_display, output_file_path, duration=4, media_path=sample_media_path)


STDOUT:

STDERR:
ffmpeg version 7.0.2 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 15.0.0 (clang-1500.3.9.4)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.0.2_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg -

In [19]:
"/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_glid.mp4"  # Optional path to the video, image, or GIF background

'/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_glid.mp4'

In [65]:
import subprocess
import os

def preprocess_videos(media_files, temp_folder):
    """
    Preprocess all videos to ensure they have the same codec and frame rate while preserving the original resolution.

    Args:
        media_files (list of str): List of media file paths to preprocess.
        temp_folder (str): Path to save the preprocessed files.
    """
    preprocessed_files = []
    for i, file_path in enumerate(media_files):
        output_file = os.path.join(temp_folder, f"preprocessed_{i}.mp4")
        # Get the original resolution of the video
        width, height = get_media_dimensions(file_path).split('x')
        command = [
            'ffmpeg',
            '-i', file_path,
            '-r', '30',  # Standardize frame rate to 30 fps
            '-s', f'{width}x{height}',  # Preserve original resolution
            '-c:v', 'libx264',
            '-pix_fmt', 'yuv420p',
            '-preset', 'fast',
            '-crf', '23',
            output_file
        ]
        subprocess.run(command, check=True)
        preprocessed_files.append(output_file)
    return preprocessed_files

def get_media_dimensions(media_path):
    try:
        command = [
            'ffprobe',
            '-v', 'error',
            '-select_streams', 'v:0',
            '-show_entries', 'stream=width,height',
            '-of', 'json',
            media_path
        ]
        result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        output = json.loads(result.stdout)
        width = output['streams'][0]['width']
        height = output['streams'][0]['height']
        return f"{width}x{height}"
    except Exception as e:
        print(f"An error occurred while getting media dimensions: {e}")
        return "640x480"  # Default resolution in case of an error

def merge_and_convert_files(media_files, output_full_path):
    """
    Merge preprocessed media files into a single file and convert to both MP4 and GIF.
    Automatically extracts the base path from the provided full output path to generate outputs.
    """
    temp_folder = "/tmp"
    preprocessed_files = preprocess_videos(media_files, temp_folder)

    # Extract base path and extension for output files
    output_base, output_ext = os.path.splitext(output_full_path)

    # Write the file list to a text file for ffmpeg processing
    with open('filelist.txt', 'w') as file:
        for preprocessed_file in preprocessed_files:
            file.write(f"file '{preprocessed_file}'\n")

    # Output file for concatenated MP4
    output_mp4 = f"{output_base}.mp4"
    if os.path.exists(output_mp4):
        os.remove(output_mp4)

    

    # Concatenate videos for MP4 output
    concat_command = [
        'ffmpeg',
        '-f', 'concat',
        '-safe', '0',
        '-i', 'filelist.txt',
        '-c', 'copy',
        output_mp4
    ]
    subprocess.run(concat_command, check=True)

    # Convert the output MP4 to GIF for a unified output
    output_gif = f"{output_base}.gif"
    if os.path.exists(output_gif):
        os.remove(output_gif)
        
    gif_command = [
        'ffmpeg',
        '-i', output_mp4,
        '-vf', "fps=10,scale=320:-1:flags=lanczos",
        '-loop', '0',
        output_gif
    ]
    subprocess.run(gif_command, check=True)

    # Optionally, clean up temporary files
    for file in preprocessed_files:
        os.remove(file)
    os.remove('filelist.txt')

# Usage example
video_list = [
    "/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_in_h.mp4",
    "/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_in_a.mp4",
    "/Users/hims/Downloads/CogVideo/black_mercesdese_benz_suv_and_.mp4",
    "/Users/hims/Downloads/CogVideo/a_black_mercedes_benz_suv_glid.mp4",
    "/tmp/output_video.mp4"
]

output_file = "/Users/hims/Downloads/CogVideo/merc_merged_video.mp4"
merge_and_convert_files(video_list, output_file)


ffmpeg version 7.0.2 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 15.0.0 (clang-1500.3.9.4)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.0.2_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex 

In [64]:
video_list = ["/Users/hims/Downloads/Tune-A-Video/a_cat_is_playing_in_grass.gif", 
              "/Users/hims/Downloads/Tune-A-Video/scene_1___cat_is_playing_in_gr.gif",
              "/tmp/output_video.gif"]  # Replace with your video file paths

output_file = "/Users/hims/Downloads/Tune-A-Video/merged.mp4"
merge_and_convert_files(video_list, output_file)


ffmpeg version 7.0.2 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 15.0.0 (clang-1500.3.9.4)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.0.2_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex 

In [72]:
import re
models_output_path = "/home/ubuntu/videos/"

def get_file_name(model_name, query, ext, final=True):
    
    model_path = f"{models_output_path}/{model_name}"
    filename = re.sub(r'[^a-zA-Z0-9]', '_', query.strip()).lower()[:47]
    if final:
        filename = f"op_{filename}"
    
    path = f"{models_output_path}/{model_name}/{filename}.{ext}"
    query_file_path = f"{models_output_path}/{model_name}/{filename}.txt"
    
        
    print(f"For {query = }, {path = }, {query_file_path = }")
    return path


def get_prompts(prompts):
    
    prompts_list =  prompts.split("\n")
    prompts_list = list(filter(lambda x: x.strip() != "", prompts_list))
    print(f"{prompts_list = }")
    return prompts_list


ps = get_prompts("""a black mercedes benz suv in hilly area going off road. There are forest and mountains around it.

a black mercedes benz suv in a city going through roads in comfortable manner. It shows the multiple building behind the scene.

a black mercedes benz suv gliding on highway in full speed.

summarize with  Merscdese Benz \n Sheer Driving Pleasure""")


for p in ps:
    get_file_name("CogVideo", p, "mp4", False)
    print("\n\n")


prompts_list = ['a black mercedes benz suv in hilly area going off road. There are forest and mountains around it.', 'a black mercedes benz suv in a city going through roads in comfortable manner. It shows the multiple building behind the scene.', 'a black mercedes benz suv gliding on highway in full speed.', 'summarize with  Merscdese Benz ', ' Sheer Driving Pleasure']
For query = 'a black mercedes benz suv in hilly area going off road. There are forest and mountains around it.', path = '/home/ubuntu/videos//CogVideo/a_black_mercedes_benz_suv_in_hilly_area_going_o.mp4', query_file_path = '/home/ubuntu/videos//CogVideo/a_black_mercedes_benz_suv_in_hilly_area_going_o.txt'



For query = 'a black mercedes benz suv in a city going through roads in comfortable manner. It shows the multiple building behind the scene.', path = '/home/ubuntu/videos//CogVideo/a_black_mercedes_benz_suv_in_a_city_going_throu.mp4', query_file_path = '/home/ubuntu/videos//CogVideo/a_black_mercedes_benz_suv_in_a_ci

In [ ]:
for 